# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the `mlcroissant` library, following the Croissant schema.

### Dataset Source
The dataset is described by a Croissant schema and available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata (as an object)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, their `@id`s, and get a schema overview.

In [ ]:
# List all record sets with their @id and name/description
print("Available record sets:")
for rs in metadata.record_sets:
    print(f"- @id: {rs.id}\n  Name: {rs.name}\n  Description: {rs.description}\n")

# For each record set, list available fields/columns by their @id and name
for rs in metadata.record_sets:
    print(f"Fields in recordSet '@id': {rs.id} ({rs.name})")
    for field in rs.fields:
        print(f"    - @id: {field.id}\n      name: {field.name}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
The record set and field `@id`s below are taken from the data overview.

In [ ]:
# Let's collect all main record set IDs
record_set_ids = [rs.id for rs in metadata.record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}

# For demonstration, let's load the first record set (update this if you wish to use a particular one)
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set: {record_set_id}")

# Pick the first available record set for demonstration
if len(record_set_ids) > 0:
    main_record_set_id = record_set_ids[0]
    df = dataframes[main_record_set_id]
    print(f"\nColumns in DataFrame for '{main_record_set_id}':\n", df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and analyze the numeric columns using their field `@id`s.

**Note:** Replace field `@id`s below with those relevant to your exploration by referring to the overview above.

In [ ]:
# Example: Select a numeric field for analysis.

# Show available columns for selection again
print("Columns in selected DataFrame:", df.columns.tolist())

# You may need to adjust these field @ids based on the field list above
# Let's try to automatically choose a numeric column
numeric_field = None
for col in df.columns:
    # Naively select a numeric field, can refine further using metadata
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
if numeric_field:
    print(f"Using column '{numeric_field}' for numeric analysis.")
    threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'bool' else 1
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt grouping by a (non-numeric) field
    group_field = None
    for col in df.columns:
        if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
else:
    print("No numeric field found in the current record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load, inspect, and analyze the FAIR² protein mass spectrometry dataset.

- We reviewed dataset structure by `@id`, loaded records into pandas DataFrames, and performed simple numeric and grouping analyses.
- For further exploration, substitute `record set` and `field @id`s above with your chosen schema elements as needed.